In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from datetime import datetime

In [2]:
def parse_log_line(line):
    pattern = r'(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}).*?(\w+)\[(\d+)\]: (.+)'
    match = re.match(pattern, line)
    if match:
        return {
            'timestamp': match.group(1),
            'service': match.group(2),
            'pid': match.group(3),
            'message': match.group(4)
        }
    return None

# Load the log file
lines = []
with open('/var/log/auth.log', 'r') as f:
    for line in f:
        parsed = parse_log_line(line.strip())
        if parsed:
            lines.append(parsed)

df = pd.DataFrame(lines)
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(df.shape)
print(df.head())

(187, 4)
            timestamp service  pid  \
0 2026-05-10 12:33:38  logind  202   
1 2026-05-10 12:33:43   login  391   
2 2026-05-10 12:33:43   login  391   
3 2026-05-10 12:33:44   login  391   
4 2026-05-10 12:33:44  logind  202   

                                             message  
0                                    New seat seat0.  
1  PAM unable to dlopen(pam_lastlog.so): /usr/lib...  
2           PAM adding faulty module: pam_lastlog.so  
3  pam_unix(login:session): session opened for us...  
4                        New session 1 of user root.  


In [3]:
print(df['service'].value_counts())

service
CRON        68
logind      46
login       38
usermod     20
polkitd      8
groupadd     3
useradd      1
passwd       1
chfn         1
gpasswd      1
Name: count, dtype: int64


In [4]:
# Filter for security-relevant events
security_keywords = ['failed', 'failure', 'invalid', 'error', 'sudo', 'root', 'session opened', 'session closed']

def flag_security(message):
    message_lower = message.lower()
    for keyword in security_keywords:
        if keyword in message_lower:
            return keyword
    return 'normal'

df['event_type'] = df['message'].apply(flag_security)
print(df['event_type'].value_counts())

event_type
normal            100
root               75
session opened     10
sudo                2
Name: count, dtype: int64
